# Week 3 — Information Security in Centralized Training Models
### Trustworthy AI · Attacks, Defenses & Secure ML (PyTorch demos)

This notebook illustrates the core ideas from the Week 3 slides on a single, centrally-trained
image classifier. In centralized training all data and computation live on one server — efficient,
but a single point of failure. We demonstrate the main **attacks** on such a model and the matching
**defenses**, then touch on **secure-ML** techniques.

**Roadmap**
1. Setup — a small centrally-trained CNN on MNIST (the "victim" model)
2. *Attack:* Data poisoning (label-flipping + backdoor trigger)
3. *Defense:* Data sanitization / anomaly detection
4. *Attack:* Evasion / adversarial examples (FGSM)
5. *Defense:* Adversarial training
6. *Attack:* Membership inference (privacy leak)
7. *Defense:* Differential privacy (DP-SGD style noise)
8. *Attack:* Model extraction (stealing the decision boundary)
9. *Defense:* Query limiting / output perturbation
10. Secure-ML wrap-up

> **Note:** cells are written to run top-to-bottom on CPU or GPU, but nothing is executed for you —
> run them yourself. Everything uses **PyTorch**.


## 0. Setup
A central server trains one CNN on MNIST. This is our victim model for every attack below.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets, transforms
import numpy as np
import copy
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

tfm = transforms.Compose([transforms.ToTensor()])
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

# Keep it small so everything runs fast on CPU
train_ds = Subset(train_full, range(6000))
test_ds  = Subset(test_full,  range(2000))


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def train_model(dataset, epochs=2, lr=1e-3, model=None, verbose=True):
    model = model or SmallCNN()
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)
    model.train()
    for ep in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            opt.step()
        if verbose:
            print(f"  epoch {ep+1}/{epochs}  loss={loss.item():.3f}")
    return model


@torch.no_grad()
def accuracy(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=256)
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item()
        total += len(yb)
    return correct / total


print("Training the clean central model...")
clean_model = train_model(train_ds, epochs=2)
print("Clean test accuracy:", round(accuracy(clean_model, test_ds), 4))


## 1. Data Poisoning Attacks
Adversaries inject malicious samples into the training set to manipulate model behavior.
We show the two types from the slides:

- **Label-flipping** — flip correct labels to wrong ones.
- **Backdoor** — stamp a small trigger patch onto some images and relabel them to a target class.
  The model behaves normally on clean data but misclassifies *any* triggered input.


In [ ]:
# Materialize the small training set into tensors so we can tamper with it
def to_tensors(ds):
    xs = torch.stack([ds[i][0] for i in range(len(ds))])
    ys = torch.tensor([ds[i][1] for i in range(len(ds))])
    return xs, ys

X_train, y_train = to_tensors(train_ds)
X_test,  y_test  = to_tensors(test_ds)

# ---- Label-flipping: flip a fraction of 7s -> 1s ----
def label_flip(X, y, src=7, dst=1, frac=0.5):
    X, y = X.clone(), y.clone()
    idx = (y == src).nonzero(as_tuple=True)[0]
    n = int(len(idx) * frac)
    flip = idx[torch.randperm(len(idx))[:n]]
    y[flip] = dst
    return X, y, len(flip)

# ---- Backdoor: white 3x3 patch in the corner -> label 0 ----
TARGET = 0
def add_trigger(x):
    x = x.clone()
    x[:, -3:, -3:] = 1.0          # bright square, bottom-right
    return x

def backdoor_poison(X, y, frac=0.1, target=TARGET):
    X, y = X.clone(), y.clone()
    n = int(len(X) * frac)
    idx = torch.randperm(len(X))[:n]
    X[idx] = add_trigger(X[idx])
    y[idx] = target
    return X, y, idx

Xp, yp, n_flip = label_flip(X_train, y_train)
Xp, yp, bd_idx = backdoor_poison(Xp, yp, frac=0.1)
print(f"Flipped {n_flip} labels, stamped {len(bd_idx)} backdoor samples")

poison_ds = TensorDataset(Xp, yp)
poisoned_model = train_model(poison_ds, epochs=2, verbose=False)

# Attack success rate: put trigger on ALL clean test images, measure -> TARGET
X_trig = add_trigger(X_test)
with torch.no_grad():
    poisoned_model.eval()
    trig_pred = poisoned_model(X_trig.to(DEVICE)).argmax(1).cpu()
asr = (trig_pred == TARGET).float().mean().item()

print("Clean test acc (poisoned model): ", round(accuracy(poisoned_model, test_ds), 4))
print("Backdoor attack success rate:    ", round(asr, 4), "<- high = attack works, accuracy looks fine")


### Visualizing the backdoor trigger

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.5))
ax[0].imshow(X_test[0,0], cmap="gray"); ax[0].set_title("clean"); ax[0].axis("off")
ax[1].imshow(X_trig[0,0], cmap="gray"); ax[1].set_title("triggered"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## 2. Defense — Data Sanitization / Anomaly Detection
Detect and remove suspicious samples *before* training. A simple detector: fit a per-class feature
statistic and flag samples whose activations are far from their class centroid (poisoned/backdoored
samples tend to be outliers). Slides also list robust training and provenance checks.


In [ ]:
# Use the clean model's penultimate features as an embedding
feat_model = copy.deepcopy(clean_model).to(DEVICE)

@torch.no_grad()
def features(model, X):
    model.eval()
    x = F.max_pool2d(F.relu(model.conv1(X.to(DEVICE))), 2)
    x = F.max_pool2d(F.relu(model.conv2(x)), 2)
    x = x.flatten(1)
    return F.relu(model.fc1(x)).cpu()

emb = features(feat_model, Xp)

# Per-class centroid distance; flag the top 'frac' most distant as suspicious
flag = torch.zeros(len(Xp), dtype=torch.bool)
for cls in range(10):
    idx = (yp == cls).nonzero(as_tuple=True)[0]
    if len(idx) < 5: continue
    centroid = emb[idx].mean(0)
    dist = (emb[idx] - centroid).norm(dim=1)
    thr = dist.quantile(0.90)          # drop the farthest 10% per class
    flag[idx[dist > thr]] = True

# How many actual backdoor samples did we catch?
caught = flag[bd_idx].float().mean().item()
print(f"Flagged {flag.sum().item()} samples; caught {caught:.0%} of backdoor injections")

keep = ~flag
clean_ds2 = TensorDataset(Xp[keep], yp[keep])
sanitized_model = train_model(clean_ds2, epochs=2, verbose=False)

with torch.no_grad():
    sanitized_model.eval()
    asr2 = (sanitized_model(X_trig.to(DEVICE)).argmax(1).cpu() == TARGET).float().mean().item()
print("Backdoor ASR after sanitization:", round(asr2, 4), "(lower is better)")


## 3. Evasion / Adversarial Examples (FGSM)
Attackers add a small perturbation to an input at **inference time** so the model misclassifies it,
while a human sees no change. We implement the Fast Gradient Sign Method:
$x_{adv} = x + \epsilon \cdot \mathrm{sign}(\nabla_x \, \mathcal{L})$.


In [ ]:
def fgsm(model, X, y, eps=0.15):
    model.eval()
    X = X.clone().to(DEVICE).requires_grad_(True)
    y = y.to(DEVICE)
    loss = F.cross_entropy(model(X), y)
    loss.backward()
    X_adv = (X + eps * X.grad.sign()).clamp(0, 1).detach()
    return X_adv.cpu()

# Take a clean, correctly-classified batch
batch_X, batch_y = X_test[:256], y_test[:256]
X_adv = fgsm(clean_model, batch_X, batch_y, eps=0.15)

with torch.no_grad():
    clean_model.eval()
    acc_clean = (clean_model(batch_X.to(DEVICE)).argmax(1).cpu() == batch_y).float().mean().item()
    acc_adv   = (clean_model(X_adv.to(DEVICE)).argmax(1).cpu()   == batch_y).float().mean().item()
print(f"Accuracy on clean batch: {acc_clean:.3f}")
print(f"Accuracy on FGSM batch:  {acc_adv:.3f}  <- collapses under tiny perturbation")

fig, ax = plt.subplots(1, 3, figsize=(6, 2.5))
ax[0].imshow(batch_X[0,0], cmap="gray"); ax[0].set_title("clean"); ax[0].axis("off")
ax[1].imshow((X_adv[0,0]-batch_X[0,0]), cmap="gray"); ax[1].set_title("perturbation"); ax[1].axis("off")
ax[2].imshow(X_adv[0,0], cmap="gray"); ax[2].set_title("adversarial"); ax[2].axis("off")
plt.tight_layout(); plt.show()


## 4. Defense — Adversarial Training
Train on adversarial examples so the model learns to resist them. Slides also list input
preprocessing (denoising, feature squeezing), certified defenses, and detection systems.


In [ ]:
def adversarial_train(dataset, epochs=2, eps=0.15):
    model = SmallCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            # generate adversarial version of the batch
            xb_adv = xb.clone().requires_grad_(True)
            loss = F.cross_entropy(model(xb_adv), yb)
            grad = torch.autograd.grad(loss, xb_adv)[0]
            xb_adv = (xb + eps * grad.sign()).clamp(0, 1).detach()
            # train on both clean + adversarial
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb) + F.cross_entropy(model(xb_adv), yb)
            loss.backward(); opt.step()
    return model

robust_model = adversarial_train(train_ds, epochs=2)
X_adv_r = fgsm(robust_model, batch_X, batch_y, eps=0.15)
with torch.no_grad():
    robust_model.eval()
    acc_adv_r = (robust_model(X_adv_r.to(DEVICE)).argmax(1).cpu() == batch_y).float().mean().item()
print(f"Robust model accuracy under FGSM: {acc_adv_r:.3f}  (vs {acc_adv:.3f} undefended)")


## 5. Membership Inference (Privacy Leak)
Can an attacker tell whether a specific sample was in the training set? A model is usually more
**confident** on data it memorized. A threshold on prediction confidence separates members from
non-members — a privacy risk under GDPR for sensitive data.


In [ ]:
@torch.no_grad()
def max_confidence(model, X):
    model.eval()
    probs = F.softmax(model(X.to(DEVICE)), dim=1)
    return probs.max(1).values.cpu()

# members = training data the model saw; non-members = held-out test data
members     = max_confidence(clean_model, X_train[:1000])
non_members = max_confidence(clean_model, X_test[:1000])

# A simple confidence-threshold attack
thr = 0.9
tp = (members > thr).float().mean().item()      # members correctly identified
fp = (non_members > thr).float().mean().item()  # non-members wrongly flagged
print(f"Members  flagged as 'in training': {tp:.2%}")
print(f"Non-members flagged as 'in training': {fp:.2%}")
print(f"Attack advantage (TP - FP): {tp - fp:.2%}  (>0 means privacy leaks)")

plt.figure(figsize=(5,2.5))
plt.hist(members.numpy(),     bins=30, alpha=0.6, label="members")
plt.hist(non_members.numpy(), bins=30, alpha=0.6, label="non-members")
plt.legend(); plt.xlabel("max softmax confidence"); plt.title("Membership signal"); plt.show()


## 6. Defense — Differential Privacy (DP-SGD style)
Add clipped-gradient noise during training so the result does not reveal any single data point.
This is a lightweight illustration of DP-SGD (per-sample clipping + Gaussian noise), adopted in
production at Google and Apple. It trades a little utility for privacy.


In [ ]:
def dp_train(dataset, epochs=2, lr=1e-3, clip=1.0, noise_mult=1.0):
    model = SmallCNN().to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            # clip + add Gaussian noise to gradients (simplified DP-SGD)
            for p in model.parameters():
                if p.grad is None: continue
                torch.nn.utils.clip_grad_norm_(p, clip)
                p.grad += torch.randn_like(p.grad) * noise_mult * clip / len(yb)
            opt.step()
    return model

dp_model = dp_train(train_ds, epochs=2, noise_mult=1.0)
m2  = max_confidence(dp_model, X_train[:1000])
nm2 = max_confidence(dp_model, X_test[:1000])
adv2 = (m2 > thr).float().mean().item() - (nm2 > thr).float().mean().item()
print("DP model test accuracy:", round(accuracy(dp_model, test_ds), 4))
print(f"Membership attack advantage with DP: {adv2:.2%}  (closer to 0 = more private)")


## 7. Model Extraction (Stealing the Model)
An adversary with only **query access** labels their own inputs using the victim's predictions,
then trains a *clone* that approximates the decision boundary — IP theft.


In [ ]:
# Attacker queries the victim on unlabeled data and trains on the (input, victim-label) pairs
query_X = X_test[:1500]
with torch.no_grad():
    clean_model.eval()
    stolen_labels = clean_model(query_X.to(DEVICE)).argmax(1).cpu()

steal_ds = TensorDataset(query_X, stolen_labels)
clone_model = train_model(steal_ds, epochs=3, verbose=False)

# Agreement between clone and victim on fresh data
probe = X_test[1500:2000]
with torch.no_grad():
    v = clean_model(probe.to(DEVICE)).argmax(1).cpu()
    c = clone_model(probe.to(DEVICE)).argmax(1).cpu()
print(f"Clone/victim agreement: {(v==c).float().mean().item():.2%}  <- model successfully extracted")


## 8. Defense — Query Limiting + Output Perturbation
Reduce extraction success by returning only coarse outputs (top-1, no probabilities), rate-limiting
queries, and adding small noise to outputs. Here we perturb the labels the victim hands back.


In [ ]:
def defended_query(model, X, wrong_rate=0.15):
    with torch.no_grad():
        model.eval()
        lbl = model(X.to(DEVICE)).argmax(1).cpu()
    # randomly corrupt a fraction of returned labels
    mask = torch.rand(len(lbl)) < wrong_rate
    lbl[mask] = torch.randint(0, 10, (mask.sum(),))
    return lbl

noisy_labels = defended_query(clean_model, query_X, wrong_rate=0.15)
steal_ds2 = TensorDataset(query_X, noisy_labels)
clone2 = train_model(steal_ds2, epochs=3, verbose=False)
with torch.no_grad():
    c2 = clone2(probe.to(DEVICE)).argmax(1).cpu()
print(f"Clone/victim agreement WITH defense: {(v==c2).float().mean().item():.2%}  (lower = harder to steal)")


## 9. Secure ML — Wrap-up
Secure ML means building **security by design** into the whole lifecycle rather than patching
vulnerabilities after the fact. Techniques from the slides:

| Technique | What it protects | Shown here |
|---|---|---|
| Data sanitization / robust training | Training data integrity | §2 |
| Adversarial training / certified defenses | Inference-time robustness | §4 |
| Differential privacy | Individual data points | §6 |
| Query limiting / output perturbation | Model IP | §8 |
| Federated learning | Central data leakage | see Week 4 notebook |
| Homomorphic encryption / MPC | Data during computation | conceptual below |

**Homomorphic encryption & MPC (conceptual):** compute on *encrypted* data without decrypting, so
raw data never appears in the clear — enabling e.g. cross-hospital training where no institution
shares raw records. These are expensive and usually integrated via specialized libraries rather
than hand-rolled.

**Takeaways**
- Centralized training is powerful but a single point of failure.
- Attacks span **data** (poisoning), **model** (evasion, extraction, inference), and **infrastructure**.
- Defenses must operate at all three levels — and security is also **policy, monitoring, and governance**, not just code.
